In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

model_df = pd.read_parquet("airport_15min_model_data.parquet")

model_df["timestamp"] = pd.to_datetime(model_df["timestamp"], utc=True)
model_df = model_df.sort_values("timestamp").reset_index(drop=True)

model_df.head()

,timestamp,ins,outs,total,outs_share,slot,weekday,month,is_weekend
0,2024-04-01 00:00:00+00:00,413.0,84.0,497.0,0.169014,0,0,3,0
1,2024-04-01 00:15:00+00:00,182.0,36.0,218.0,0.165138,1,0,3,0
2,2024-04-01 00:30:00+00:00,104.0,55.0,159.0,0.345912,2,0,3,0
3,2024-04-01 00:45:00+00:00,49.0,34.0,83.0,0.409639,3,0,3,0
4,2024-04-01 01:00:00+00:00,28.0,26.0,54.0,0.481481,4,0,3,0


# Train/test split

In [2]:
split_date = model_df["timestamp"].quantile(0.8)

train_df = model_df[model_df["timestamp"] <= split_date].copy()
test_df  = model_df[model_df["timestamp"] > split_date].copy()

print(train_df["timestamp"].min(), "→", train_df["timestamp"].max(), len(train_df))
print(test_df["timestamp"].min(), "→", test_df["timestamp"].max(), len(test_df))

2024-04-01 00:00:00+00:00 → 2025-11-05 22:30:00+00:00 55289
2025-11-05 22:45:00+00:00 → 2026-03-31 23:45:00+00:00 13822


# Baseline A: global average

In [3]:
global_total_mean = train_df["total"].mean()
global_outs_share = train_df["outs_share"].mean()

test_df["pred_total_global"] = global_total_mean
test_df["pred_outs_share_global"] = global_outs_share

test_df["pred_outs_global"] = test_df["pred_total_global"] * test_df["pred_outs_share_global"]
test_df["pred_ins_global"] = test_df["pred_total_global"] - test_df["pred_outs_global"]

# Baseline B: slot-of-day average

In [4]:
slot_stats = (
    train_df
    .groupby("slot")
    .agg(
        pred_total_slot=("total", "mean"),
        pred_outs_share_slot=("outs_share", "mean")
    )
    .reset_index()
)

test_df = test_df.merge(slot_stats, on="slot", how="left")

test_df["pred_total_slot"] = test_df["pred_total_slot"].fillna(global_total_mean)
test_df["pred_outs_share_slot"] = test_df["pred_outs_share_slot"].fillna(global_outs_share)

test_df["pred_outs_slot"] = test_df["pred_total_slot"] * test_df["pred_outs_share_slot"]
test_df["pred_ins_slot"] = test_df["pred_total_slot"] - test_df["pred_outs_slot"]

# Baseline C: slot + weekday average

In [5]:
slot_weekday_stats = (
    train_df
    .groupby(["slot", "weekday"])
    .agg(
        pred_total_slot_weekday=("total", "mean"),
        pred_outs_share_slot_weekday=("outs_share", "mean")
    )
    .reset_index()
)

test_df = test_df.merge(slot_weekday_stats, on=["slot", "weekday"], how="left")

test_df["pred_total_slot_weekday"] = test_df["pred_total_slot_weekday"].fillna(test_df["pred_total_slot"])
test_df["pred_outs_share_slot_weekday"] = test_df["pred_outs_share_slot_weekday"].fillna(test_df["pred_outs_share_slot"])

test_df["pred_outs_slot_weekday"] = (
    test_df["pred_total_slot_weekday"] * test_df["pred_outs_share_slot_weekday"]
)
test_df["pred_ins_slot_weekday"] = (
    test_df["pred_total_slot_weekday"] - test_df["pred_outs_slot_weekday"]
)

# Evaluation metrics

In [6]:
def mae(y, yhat):
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))

def evaluate(prefix):
    return {
        "total_mae": mae(test_df["total"], test_df[f"pred_total_{prefix}"]),
        "total_rmse": rmse(test_df["total"], test_df[f"pred_total_{prefix}"]),
        "outs_mae": mae(test_df["outs"], test_df[f"pred_outs_{prefix}"]),
        "outs_rmse": rmse(test_df["outs"], test_df[f"pred_outs_{prefix}"]),
        "ins_mae": mae(test_df["ins"], test_df[f"pred_ins_{prefix}"]),
        "ins_rmse": rmse(test_df["ins"], test_df[f"pred_ins_{prefix}"]),
        "direction_mae": mae(test_df["outs_share"], test_df[f"pred_outs_share_{prefix}"]),
    }

results = pd.DataFrame({
    "global": evaluate("global"),
    "slot": evaluate("slot"),
    "slot_weekday": evaluate("slot_weekday"),
}).T

results

,total_mae,total_rmse,outs_mae,outs_rmse,ins_mae,ins_rmse,direction_mae
global,131.503793,160.122811,81.755828,96.869366,69.070972,84.139071,0.161458
slot,75.831100,97.738296,41.842536,55.594710,42.235025,56.325320,0.086621
slot_weekday,64.074949,83.278428,35.838424,48.080332,37.193288,50.036198,0.082968


In [7]:
def direction_accuracy(prefix):
    true_direction = test_df["outs"] > test_df["ins"]
    pred_direction = test_df[f"pred_outs_{prefix}"] > test_df[f"pred_ins_{prefix}"]
    return np.mean(true_direction == pred_direction)

results["direction_accuracy"] = [
    direction_accuracy("global"),
    direction_accuracy("slot"),
    direction_accuracy("slot_weekday"),
]

results

,total_mae,total_rmse,outs_mae,outs_rmse,ins_mae,ins_rmse,direction_mae,direction_accuracy
global,131.503793,160.122811,81.755828,96.869366,69.070972,84.139071,0.161458,0.590870
slot,75.831100,97.738296,41.842536,55.594710,42.235025,56.325320,0.086621,0.808060
slot_weekday,64.074949,83.278428,35.838424,48.080332,37.193288,50.036198,0.082968,0.808132


# Saving Data

In [8]:
train_df.to_parquet("airport_15min_train.parquet", index=False)
test_df.to_parquet("airport_15min_test.parquet", index=False)
results.to_csv("baseline_results.csv")